## Домашнее задание №15

* Поднять модель с помощью **vLLM** и отправить запрос — **2 балла**
* Сделать запрос с **guided JSON** — **2 балла**
* Сделать запрос с другими guided-запросами — **2 балла**
* Сделать запросы с разными параметрами генерации:
  `max_p`, `max_k`, `temperature` и др. + вывод — **4 балла**

---

Домашнее задание необходимо предоставить в формате **ссылки на Google Colab / Jupyter Notebook** с вашими действиями и ключевыми выводами.

**Мягкий дедлайн:** 15 декабря 23:59 Мск
**Жёсткий дедлайн:** 22 декабря 23:59 Мск
(Желательно не затягивать со сдачей :)

# Подготовка програмного обеспечения vLLM

## Сборка проекта из исходного кода


1. Клонируем репозиторий

```bash
git clone https://github.com/vllm-project/vllm.git
```

2. Переходим в проект vllm

```bash
cd vllm        
```

3. Устанавливаем зависимости

```bash
 uv pip install -r requirements/cpu.txt --index-strategy unsafe-best-match
 uv pip install -e .
```

4. Сборка Docker Image

```bash
docker build -f docker/Dockerfile.cpu \                                  
        --tag vllm-cpu-env .
```


### Запуск  Qwen/Qwen3-0.6B  vLLM  (Apple Sillicon)

```bash
 docker run --rm \
  --privileged=true \
  --shm-size=16g \
  -p 8000:8000 \
  -e VLLM_CPU_OMP_THREADS_BIND=4 \
  vllm-cpu-env Qwen/Qwen3-0.6B \
    --dtype float32 \
    --max-model-len 1024 \
    --host 0.0.0.0 --port 8000
```

# Экспериментируем с VLLM

In [1]:
from openai import OpenAI

In [2]:
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"


client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

## 1. Сделать запрос с **guided JSON**

In [3]:
from pydantic import BaseModel

class Info(BaseModel):
    name: str
    age: int

model = client.models.list().data[0].id
completion = client.beta.chat.completions.parse(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What's my name and age?"},
    ],
    response_format=Info,
    max_completion_tokens=512,
)

message = completion.choices[0].message
print(message)
print("Name:", message.parsed.name)
print("Age:", message.parsed.age)

ParsedChatCompletionMessage[Info](content='{"name": "Lena", "age": 15}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, parsed=Info(name='Lena', age=15), reasoning=None, reasoning_content=None)
Name: Lena
Age: 15


In [4]:
message.content

'{"name": "Lena", "age": 15}'

## 2. Сделать запрос с другими guided-запросами

In [5]:
simplified_sql_grammar = """
    root ::= select_statement

    select_statement ::= "SELECT " column " from " table " where " condition

    column ::= "col_1 " | "col_2 "

    table ::= "table_1 " | "table_2 "

    condition ::= column "= " number

    number ::= "1 " | "2 "
"""

completion = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": "Generate an SQL query to show the 'username' and 'email' from the 'users' table.",
        }
    ],
    extra_body={"structured_outputs": {"grammar": simplified_sql_grammar}},
)
print(completion.choices[0].message.content)

SELECT col_1  from table_1  where col_1 = 1 


## 3. Сделать запросы с разными параметрами генерации:

### Базовый пример запроса

In [6]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

response = client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[{"role": "user", "content": "Объясни квантовую суперпозицию"}],
)

print(response.choices[0].message.content)


<think>
Хорошо, пользователь спрашивает о квантовой суперпозиции. Нужно объяснить это понятно и избегать технических терминов, но сохранять точность. Сначала вспомню основные понятия квантовой физики. Квантовая суперпозиция — это физическая теория, которая объясняет, как частицы могут быть в нескольких состояниях одновременно. 

Надо подчеркнуть, что это не просто совместное состояние, а физическое явление. Вспомню пример из квантовой механики: электрон может быть в состоянии, где он обладает одновременно двух энергиях. Это связано с возможностью обмена состояниями между носителями, а не с непрерывным движением. 

Важно упомянуть, что суперпозиция не обязательно приводит к нулю, но может быть связано с другими явлениями, например, суперпозицией в квантовых волнах. Также стоит отметить, что это явление было предложено в 1930-х годах и связано с принципом суперпозиции, который не было до этого в классической физике. 

Нужно проверить, нет ли ошибок в описании. Возможно, стоит добавить, ч

### 1️⃣ Способ генерации

* **Метод:** стандартный вызов `client.chat.completions.create()` без указания `temperature`, `top_p` или `max_tokens` — то есть **default параметры генерации**.
* **Параметры:** только `model="Qwen/Qwen3-0.6B"` и базовый `messages`.
* **Тип вывода:** полное завершение (non-streaming), GPT-подобная генерация в стиле «обученного на объяснениях».

---

### 2️⃣ Характеристика ответа

* **Длина:** довольно длинный, детализированный ответ (~400–500 слов, включая примеры и пояснения).
* **Стиль:** образовательный, структурированный, с подзаголовками и пунктами.
* **Качество информации:** точное объяснение с базовыми примерами (электроны, квантовые волны).
* **Логическая структура:** есть постепенное введение, объяснение, примеры, значимость.
* **Использование технических терминов:** минимальное, с пояснением («состояния одновременно», «суперпозицией»).
* **Особенность:** модель вставляет внутренние размышления `<think> ... </think>`, как бы «планируя» ответ перед генерацией основной части. Это хорошо для длинного, продуманного текста.

---

### 3️⃣ Вывод по способу генерации

* **Default chat completion** без настройки `temperature`/`top_p` → **сильный, связный, образовательный текст**.
* Подходит для: объяснения научных понятий, создания длинных информативных текстов, статей, образовательного контента.
* Минусы:

  * Может быть слишком длинным для быстрых запросов.
  * Меньше контроля над стилем (креативность/разброс ответов минимален).


### Пример №1 — Управление длиной и температурой

In [7]:
response = client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[{"role": "user", "content": "Объясни квантовую суперпозицию"}],
    max_tokens=150,
    temperature=1.0,
    top_p=0.9,
)

print(response.choices[0].message.content)


<think>
Хорошо, пользователь просит объяснить квантовую суперпозицию. Начну с того, что вспомню основные понятия. Суперпозиция в кванте — это сочетание между несколькими возможностями. Например, при измерении, квантовый компьютер не получает информацию о том, какой случай он имеет, но продолжает суперпозиционное состояние.

Нужно объяснить, почему это происходит. Возможно, связать с принципом суперпозиции в квантовой механике, который не определяет конкретную состояние. Далее, дать примеры


### 1️⃣ Способ генерации

* **Метод:** стандартный `chat.completions.create()`.
* **Параметры генерации:**

  * `max_tokens=150` — ограничение длины ответа, короткая генерация.
  * `temperature=1.0` — высокая «творческая свобода», модель может выдавать более вариативные формулировки.
  * `top_p=0.9` — вероятностная выборка из топ 90% вероятных токенов, добавляет разнообразие.

---

### 2️⃣ Характеристика ответа

* **Длина:** ограниченная (~150 токенов), гораздо короче, чем в базовом примере.
* **Стиль:** менее формальный, более «разговорный» и креативный.
* **Логическая структура:** упрощённая, модель сразу пытается объяснить с примерами, но не успевает полностью завершить мысль.
* **Особенность:** `<think>` блок короче и менее детализирован; видно, что модель планирует ответ, но из-за ограничения токенов не успевает его полностью развить.
* **Качество информации:** точное, но упрощённое; термины могут быть слегка расплывчатыми, без глубокого контекста.

---

### 3️⃣ Вывод по способу генерации

* **Использование `max_tokens` + высокая `temperature` и `top_p`** → быстрый, компактный, креативный ответ.
* Подходит для: кратких объяснений, творческих ответов, вариантов формулировок.
* Минусы:

  * Меньше деталей, примеров и логической последовательности.
  * Ответ может обрываться или быть неполным из-за лимита токенов.

### Пример №2 — Детемплейтное, точное, «сухое» объяснение

In [8]:
response = client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[{"role": "user", "content": "Объясни квантовую суперпозицию"}],
    temperature=0.1,
    top_p=1.0,
)

print(response.choices[0].message.content)


<think>
Хорошо, пользователь просит объяснить квантовую суперпозицию. Начну с того, что вспомню основные понятия квантовой физики. Квантовая суперпозиция — это явление, когда частицы или системы находятся в нескольких состояниях одновременно. Это связано с квантовыми состояниями, которые не могут быть описаны однозначно. Нужно упомянуть, что это отличается от классической суперпозиции, которая включает в себя множество возможных вариантов.

Сначала объясню, что квантовая суперпозиция — это когда объект находится в нескольких состояниях. Например, электрон может быть в состоянии электронов с положением и спином, но это не означает, что он может быть в нескольких положениях одновременно. Нужно подчеркнуть, что это не означает, что он может быть в нескольких положениях, а просто что он может быть в нескольких состояниях.

Важно отметить, что квантовая суперпозиция не является классической суперпозицией. В классической физике суперпозиция означает, что объект может быть в нескольких положе

### 1️⃣ Способ генерации

* **Метод:** стандартный `chat.completions.create()`.
* **Параметры генерации:**

  * `temperature=0.1` — почти детерминированный вывод, минимальная случайность.
  * `top_p=1.0` — учитываются все вероятные токены, выбор почти полностью предсказуемый.
  * **max_tokens не указан**, поэтому модель генерирует длинный ответ до внутреннего лимита.

---

### 2️⃣ Характеристика ответа

* **Длина:** длиннее и более развернутая, чем в примере №1.
* **Стиль:** «сухой», точный, мало креативности.
* **Логическая структура:** очень четкая, хорошо организованная, с пунктами и примерами.
* **Особенность:** `<think>` блок помогает понять планирование ответа. Ответ не пытается быть художественным или «разговорным».
* **Качество информации:** высокая точность, термины аккуратные, детали квантовой суперпозиции объяснены корректно.

---

### 3️⃣ Вывод по способу генерации

* **Использование низкой `temperature` и полного `top_p`** → строгий, детерминированный, научно корректный ответ.
* Подходит для: учебных материалов, точных объяснений, технической документации.
* Минусы:

  * Меньше вариативности и креативности.
  * Может быть «сухо» и формально, менее «живой» стиль.


### Пример №3 — Высокая креативность

In [9]:
response = client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[{"role": "user", "content": "Объясни квантовую суперпозицию"}],
    temperature=1.5,
    top_p=0.95,
)

print(response.choices[0].message.content)


<think>
Хорошо, давайте уберём всё тут, но я идёт вспоминать про квантовую суперпозицию! Пожалуйста, упрости ответ. Студент науку может описать суперпозицию, а научные люди и физики вдумывются во внутренний мир. Не забудь о технических терминах и примерах. Скорее, просто говорю, это когда фунции принимают значение и нет одно. Удивительно, а потом всё становится понятнее.
</think>

Квантовая суперпозиция — это фундаментальное явление, которое позволяет частицам, например, электронам или шине, обладать несколько возможными значениями одновременно. То есть, в момент начала наблюдения, фунция (например, состояние частицы в системе) может рассматривать её в двумерах (где обе возможности останутся признанными), тогда как иначе не принимал один из них.

Она явно и привносит в природу фазовое поведение, что позволяет описывать не классический «все или нет». Это подтверждается через примеры физиков – в случае кубитов, суперпозиция — ключевой принцип, благодаря которому квантовые вычисления эффе

### 1️⃣ Способ генерации

* **Метод:** стандартный `chat.completions.create()`.
* **Параметры генерации:**

  * `temperature=1.5` — очень высокая случайность, модель склонна к креативным формулировкам.
  * `top_p=0.95` — ограничивает выбор токенов до 95% вероятности, чтобы сохранить связность, но позволяет экспериментальные варианты.
  * **max_tokens не указан**, поэтому ответ может быть длинным.

---

### 2️⃣ Характеристика ответа

* **Длина:** средняя, не слишком структурированная.
* **Стиль:** разговорный, спонтанный, «игривый».
* **Логическая структура:** свободная, фрагментарная, с местами странными формулировками (`фунции`, `в двумерах`).
* **Особенность:** `<think>` показывает, что модель активно «планирует» нестандартный ответ, пытаясь быть креативной.
* **Качество информации:** ключевая идея правильная (суперпозиция — несколько состояний одновременно), но термины смешиваются и местами вводят в заблуждение.

---

### 3️⃣ Вывод по способу генерации

* **Высокая `temperature` + `top_p<1`** → креативный, «живой», иногда художественный стиль.
* Подходит для: творческих объяснений, учебных демонстраций, вариативных формулировок.
* Минусы:

  * Менее точный, могут появляться странные формулировки или ошибки.
  * Логическая структура слабее, сложно использовать в официальных документах или учебных материалах.


### Пример №4 — Сверхстабильный вывод (nucleus sampling off)

In [10]:
response = client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[{"role": "user", "content": "Сделай краткий пересказ объяснения квантовую суперпозицию"}],
    temperature=0,
    top_p=1,
)

print(response.choices[0].message.content)


<think>
Хорошо, пользователь просит краткий пересказ объяснения квантовой суперпозиции. Начну с того, что вспомню основные моменты. Квантовая суперпозиция — это физическая теория, где квантовые системы могут находиться в нескольких состояниях одновременно. Это связано с принципом суперпозиции и волновой функции. Нужно упомянуть, что это позволяет частиц быть в нескольких состояниях, что важно для квантовой механики. Также важно отметить, что это не означает, что частицы не могут быть в разных состояниях, а просто их можно описывать в нескольких возможностях. Проверю, чтобы не было лишних деталей, и сохранил простоту и понятность.
</think>

Квантовая суперпозиция — это физическая теория, где квантовые системы могут находиться в нескольких возможных состояниях одновременно. Это связано с принципом суперпозиции и волновой функцией, позволяя частиц быть в нескольких состояниях, что важно для квантовой механики.


### 1️⃣ Способ генерации

* **Метод:** стандартный `chat.completions.create()`.
* **Параметры генерации:**

  * `temperature=0` — полностью детерминированный вывод, модель выбирает наиболее вероятные токены.
  * `top_p=1` — nucleus sampling отключен, все токены разрешены, но фактически выбор детерминирован.
  * Использован запрос на краткий пересказ, что ограничивает длину ответа.

---

### 2️⃣ Характеристика ответа

* **Длина:** короткая и компактная.
* **Стиль:** «сухой», технический, без креативных элементов.
* **Логическая структура:** чёткая и лаконичная, информация упорядочена.
* **Особенность:** `<think>` демонстрирует внутренний план модели, но в финальном выводе он минимален.
* **Качество информации:** высокая точность, нет лишних деталей, термины корректные.

---

### 3️⃣ Вывод по способу генерации

* **Температура 0 + top_p=1** → сверхстабильный, детерминированный вывод.
* Подходит для: кратких пересказов, справочных материалов, официальных документов.
* Минусы:

  * Ответ может быть «сухим» и менее наглядным.
  * Нет вариативности и креативности.


### Пример №5 — Генерация нескольких вариантов (n=3)

In [11]:
response = client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[{"role": "user", "content": "Придумай слоган для IT-компании"}],
    n=3,
    temperature=0.8,
)

for i, choice in enumerate(response.choices):
    print(f"Вариант {i+1}:", choice.message.content, "\n")


Вариант 1: <think>
Хорошо, пользователь попросил придумать слоган для IT-компании. Начну с того, что вспомню, какие ключевые элементы входят в название IT-компании. Обычно это технологии, инновации, развитие, будущее, визуализация, веб, программирование, безопасность, цифровые инновации, цифровой образование.

Нужно выбрать подходящие слова, которые можно сопоставить с брендом. Слоган должен быть современным, легко воспринимаемым и отражать ценностные аспекты компании. Может, использовать слова с сильным смыслом, например "Современство", "Будущее", "Инновации".

Также стоит учесть, что слоган должен быть в одном предложении, возможно, с несколькими частями. Например, "Современный инновационный бренд, который помогает клиентам в будущем." Но нужно убедиться, что он привлекает внимание и выделяет уникальность.

Может, использовать метафоры, чтобы сделать текст более живым. Например, "Строит будущее" или "Изменяет цифровую будущее". Нужно проверить, чтобы не было повторяющихся слов и было

### 1️⃣ Способ генерации

* **Метод:** `chat.completions.create()` с параметром `n=3` — модель создаёт сразу три независимых варианта ответа.
* **Параметры генерации:**

  * `temperature=0.8` — умеренно высокая креативность, допускается вариативность.
  * `top_p` не указан — используется значение по умолчанию (`top_p=1.0`).

---

### 2️⃣ Характеристика ответов

* **Множественные варианты:** три самостоятельных генерации, каждый с собственным подходом к формулировке слогана.
* **Креативность:** высокая — температура 0.8 даёт модели свободу выбирать менее вероятные токены.
* **Структура:**

  * Вариант 1: длинное размышление с мыслями `<think>`, затем конкретный слоган.
  * Вариант 2: несколько кратких идей, список из 10 возможных слоганов.
  * Вариант 3: комбинированный слоган с акцентом на ключевые ценности компании.
* **Особенность:** каждый вариант демонстрирует разные стили — от подробного объяснения до лаконичного креативного выражения.

---

### 3️⃣ Вывод по способу генерации

* **`n > 1` + температура >0.5** → генерация нескольких разнообразных вариантов.
* **Плюсы:**

  * Можно выбрать лучший вариант из нескольких.
  * Подходит для маркетинга, творчества, генерации идей.
* **Минусы:**

  * Больше «шума» и не всегда идеальная формулировка.
  * Требует ручного отбора из сгенерированных вариантов.

### Пример №6 — Streaming-ответы

In [12]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

for chunk in client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[{"role": "user", "content": "Напиши мини-эссе про ИИ"}],
    stream=True,
):
    delta = chunk.choices[0].delta
    if delta.content:
        print(delta.content, end="")


<think>
Хорошо, пользователь просит написать мини-эссе про ИИ. Нужно понять, что именно он хочет. Мини-эссе обычно относится к определённому теме, и здесь пользователь оставил только заголовок. Возможно, он хочет, чтобы я оставался без заголовка, а затем написал эссе по теме "ИИ". Надо убедиться, что эссе соответствует требованиям. Проверю структуру: введение, основные пункты, заключение. Важно подчеркнуть ключевые моменты ИИ, такие как его роль в современной технологии, влияние на общество, применение в различных областях. Нужно убедиться, что текст логично и не слишком длинно. Проверю на наличие ошибок и структуры. Возможно, пользователь хочет, чтобы эссе было более формальным или более разговорным, но в данном случае, как мини-эссе, должно быть академично. Убедиться, что все ключевые моменты затрагиваются, а не перегружено. Теперь приступлю к написанию.
</think>

**ИИ: Путь к будущему технологий**

ИИ (интеллектные системы) — это область исследований, которая стремится объединить ло

### 1️⃣ Способ генерации

* **Метод:** `chat.completions.create(..., stream=True)` — потоковая генерация.
* **Как работает:** модель отправляет части ответа (дельты) по мере генерации, вместо того чтобы ждать полного завершения.
* **Особенности использования:** нужно проверять наличие `delta.content`, чтобы выводить текст частями.

---

### 2️⃣ Характеристика ответа

* **Потоковая выдача:** ответ формируется постепенно; можно начать обрабатывать текст сразу.
* **Контроль над генерацией:** поток позволяет анализировать промежуточные сегменты, фильтровать или адаптировать текст «на лету».
* **Детали ответа:**

  * Имеется предварительная мыслительная секция `<think>`, где модель планирует структуру мини-эссе.
  * Основной текст организован логично: введение → основные пункты → вывод.
  * Содержится академический стиль, информативность и структурированность.

---

### 3️⃣ Преимущества

* Мгновенная отдача текста, полезно для интерфейсов, где нужна «живость» генерации.
* Позволяет прерывать генерацию или вмешиваться по ходу, если нужно изменить направление ответа.
* Удобно для длинных текстов и интерактивных приложений.

---

### 4️⃣ Минусы

* Сложнее обработка, если нужна целостная структура — нужно накапливать дельты для полного анализа.
* Больше кода для правильного объединения и отображения текста.

---

### 5️⃣ Итог

* **Streaming** — лучший вариант для интерактивных приложений, «живых» чатов или генерации больших текстов.
* Отличается от обычного запроса (`stream=False`) тем, что текст приходит частями и может использоваться динамически.

### Пример №7 — Управление длиной контекста (ноть: это работает в vLLM)

In [13]:
response = client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[{"role": "user", "content": "Продолжи текст…"}],
    max_tokens=500,
    presence_penalty=0.2,
    frequency_penalty=0.1,
)

print(response.choices[0].message.content)


<think>
Хорошо, пользователь просит продолжить текст. Нужно понять, что именно нужно продолжить. Возможно, текст был задан на русском языке и нужно продолжить его. Поскольку я не могу видеть исходный текст, я предполагаю, что пользователь хочет продолжить какой-то текст, который уже был предоставлен. Если я не могу точно определить, что именно требуется, я должен сначала запросить дополнительные данные. Возможно, пользователь не предоставил текст, и мне нужно подчеркнуть, что это необходимо. Также стоит упомянуть, что продолжение текста может включать разные аспекты, например, исторический, научный, эмоциональный или логический контекст. Надо быть внимательным и предложить помощь, чтобы уточнить.
</think>

Хорошо, пожалуйста, предоставьте текст, и я помогу продолжить его. Если вы хотите продолжить текст о научных исследованиях, истории, литературе или другом теме, укажите конкретно, чтобы я мог помочь.



### 1️⃣ Способ генерации

* **Метод:** `chat.completions.create(...)` с параметрами:

  * `max_tokens=500` — ограничение на количество токенов в ответе.
  * `presence_penalty=0.2` — стимулирует модель включать новые идеи и слова, которых ранее не было.
  * `frequency_penalty=0.1` — слегка снижает повторение уже использованных токенов.
* **Особенности vLLM:** позволяет эффективно управлять длиной ответа и «контекстом» без перегрузки памяти.

---

### 2️⃣ Характеристика ответа

* **Контекстная гибкость:** модель пытается определить, какой текст продолжить, и предупреждает, если исходный текст отсутствует.
* **Детальная мыслительная секция `<think>`:** модель объясняет, что нужно уточнение контекста, прежде чем продолжать текст.
* **Структурированность:** ответ показывает план работы модели — сначала анализ запроса, затем уточнение, затем генерация (если предоставлен текст).

---

### 3️⃣ Преимущества

* Позволяет управлять длиной ответа (`max_tokens`) для экономии ресурсов и предсказуемости.
* Параметры `presence_penalty` и `frequency_penalty` помогают избегать повторов и стимулируют креативность.
* Подходит для продолжения текстов, где важно сохранить логическую связность с предыдущим контентом.

---

### 4️⃣ Минусы

* Без предоставленного исходного текста модель не может продолжить содержательно — нужно взаимодействие с пользователем.
* Риск «слишком безопасных» ответов, если `presence_penalty` и `frequency_penalty` установлены слишком низко.

---

### 5️⃣ Итог

* Этот способ полезен, когда нужно **контролировать длину и структуру** генерируемого текста.
* Особенно подходит для vLLM, где управление контекстом и памятью более гибкое.
* Рекомендуется использовать совместно с предварительным уточнением контекста, чтобы модель понимала, что продолжать.

## Общие выводы


| Пример                                               | Параметры                                                                 | Стиль ответа                                            | Стабильность                                              | Креативность                                                                      | Особенности / комментарий                                                                        |
| ---------------------------------------------------- | ------------------------------------------------------------------------- | ------------------------------------------------------- | --------------------------------------------------------- | --------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------ |
| **Базовый пример**                                   | `temperature=1.0` по умолчанию, `top_p=1.0`, без ограничения `max_tokens` | Подробное объяснение с элементами размышления `<think>` | Средняя — ответ логичный, но длинный                      | Средняя — есть «мыслительная» часть, но структура предсказуемая                   | Полный, почти учебный стиль; демонстрирует работу механизма `<think>`                            |
| **Пример №1 — Управление длиной и температурой**     | `max_tokens=150`, `temperature=1.0`, `top_p=0.9`                          | Короткий, живой, более разговорный                      | Средняя — короткий ответ, не всегда полный                | Высокая — генерация идей ограничена токенами, появляются неожиданные формулировки | Позволяет контролировать длину ответа и случайность за счёт `temperature`                        |
| **Пример №2 — Детемплейтное объяснение**             | `temperature=0.1`, `top_p=1.0`                                            | Точный, «сухой», научный                                | Высокая — повторяемость и предсказуемость                 | Низкая — минимальные отклонения от фактов                                         | Отлично подходит для точного описания понятий, без лишних слов                                   |
| **Пример №3 — Высокая креативность**                 | `temperature=1.5`, `top_p=0.95`                                           | Экспрессивный, «весёлый», местами неформальный          | Низкая — возможны ошибки и неточности                     | Очень высокая — неожиданные формулировки, метафоры, необычные примеры             | Используется для креативных задач, но теряется точность и стабильность                           |
| **Пример №4 — Сверхстабильный вывод**                | `temperature=0`, `top_p=1.0`                                              | Короткий, сухой, лаконичный                             | Очень высокая — всегда почти одинаковый ответ             | Низкая — генерация максимально детерминирована                                    | Идеально для кратких резюме или инструкций, где важна стабильность                               |
| **Пример №5 — Генерация нескольких вариантов (n=3)** | `temperature=0.8`, `n=3`                                                  | Разные формулировки, но тематически связанные           | Средняя — зависит от `temperature` и `n`                  | Высокая — модель предлагает несколько креативных вариантов                        | Позволяет получить несколько идей за один запрос, удобно для маркетинга, креативных задач        |
| **Пример №6 — Streaming-ответы**                     | `stream=True`, стандартные `temperature` и `top_p`                        | Пошаговое формирование текста, академично               | Средняя — стабильность зависит от генерации каждого чанка | Средняя — частично ограничена                                                     | Позволяет выводить текст по мере генерации, удобно для больших эссе или интерактивных приложений |
| **Пример №7 — Управление длиной контекста**          | `max_tokens=500`, `presence_penalty=0.2`, `frequency_penalty=0.1`         | Контекстно-ориентированное, уточняющее                  | Высокая при наличии контекста                             | Средняя — немного стимулирует новые идеи                                          | Отлично для продолжения текста и управления связностью; важно предоставлять исходный контекст    |

---

### 🔹 Выводы по стратегиям генерации

1. **Контроль креативности:**

   * `temperature` — главный фактор: низкая → детерминированный стиль, высокая → креативность и неожиданные формулировки.
   * `top_p` регулирует вероятность выбора токенов — снижает риск редких слов при меньшем `top_p`.

2. **Контроль стабильности:**

   * `temperature=0` → максимально стабильный, предсказуемый текст.
   * Использование `stream=True` и `max_tokens` позволяет управлять размером и потоковой генерацией текста.

3. **Контроль длины:**

   * `max_tokens` ограничивает объём ответа.
   * `presence_penalty` и `frequency_penalty` помогают уменьшить повторения и стимулируют включение новых идей.

4. **Креативные задачи:**

   * Высокие `temperature` и `top_p<1` + `n>1` → несколько разнообразных вариантов (идеально для маркетинга, слоганов, творческих эссе).

5. **Учебные или точные ответы:**

   * `temperature≤0.1`, `top_p≈1` → короткий, ясный, научный или детальный стиль с минимальными отклонениями.

